# Задание Pro

Решите задачу по распознаванию позитивных и негативных отзывы людей по автомобилю Tesla. База для обучения содержит два текстовых файла с рядом строчных отзывов с мнением людей об автомобиле Tesla, соответственно негативного и позитивного содержания. Ссылка на скачивание базы уже включена в ноутбук задания.


Необходимо выполнить следующие действия:

  1. Загрузите саму базу по ссылке и подговьте файлы базы для обработки.
  2. Создайте обучающую и проверочную выборки, обратив особое внимание на балансировку базы: количество примеров каждого класса должно быть примерно одного порядка.
  3. Подготовьте выборки для обучения и обучите сеть. Добейтесь результата точности сети в 85-90% на проверочной выборке.
   


**Версия ноутбука переписана на PyTorch: TensorFlow/Keras не используется.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gdown
import os
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

%matplotlib inline

# Фиксация случайности для повторяемости результата
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Выбор устройства: видеокарта, если доступна, иначе процессор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)


In [ ]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/tesla.zip', None, quiet=True)

In [ ]:
!unzip -qo tesla.zip -d tesla/
!ls tesla

In [ ]:
# Функция для чтения файла
def read_text(file_name):

  read_file = open(file_name, 'r', encoding='utf-8')
  text = read_file.read()

  text = text.replace("\n", " ")

  return text

class_names = ["Негативный отзыв", "Позитивный отзыв"]

num_classes = len(class_names)

In [ ]:
# Создаём список под тексты для обучающей выборки
texts_list = []

# В Google Colab архив обычно распаковывается в /content/tesla,
# локально — в папку tesla рядом с ноутбуком.
base_dir = '/content/tesla' if os.path.exists('/content/tesla') else 'tesla'

for file_name in sorted(os.listdir(base_dir)):
    file_path = os.path.join(base_dir, file_name)

    if os.path.isfile(file_path):
        texts_list.append(read_text(file_path))
        print(file_name, 'добавлен в обучающую выборку')


In [ ]:
# Узнаем объём каждого текста в словах и символах
texts_len = [len(text) for text in texts_list]

t_num = 0

print(f'Размеры текстов по порядку (в токенах):')

# Циклом проводим итерацию по списку с объёмами текстов
for text_len in texts_len:

  t_num += 1

  print(f'Текст №{t_num}: {text_len}')

In [ ]:
# Создаём список
train_len_shares = [(i - round(i/5)) for i in texts_len]

t_num = 0

# Циклом проводим итерацию по списку с объёмами текстов
for train_len_share in train_len_shares:

  t_num += 1

  print(f'Доля 80% от текста №{t_num}: {train_len_share} символов')

In [ ]:
from itertools import chain

# списки для обучающих текстов
train_texts = []
test_texts = []

for i in range(num_classes):

  train_texts.append(texts_list[i][:train_len_shares[i]])

  test_texts.append(texts_list[i][train_len_shares[i]:])

# проверка размеров выборок
for i in range(num_classes):
  print(class_names[i])
  print('Обучающий текст:', len(train_texts[i]))
  print('Проверочный текст:', len(test_texts[i]))
  print()

In [ ]:
# параметры
vocab_size = 20000
win_size = 1000
win_hop = 100

class SimpleTokenizer:
    """Минимальный токенизатор вместо tensorflow.keras.preprocessing.text.Tokenizer."""

    def __init__(self, num_words=20000, lower=True, oov_token='unknown'):
        self.num_words = num_words
        self.lower = lower
        self.oov_token = oov_token
        self.word_index = {oov_token: 1}

    def _text_to_words(self, text):
        if self.lower:
            text = text.lower()

        text = text.replace('\xa0', ' ')
        text = re.sub(r'[!"#$%&()*+,-./:;<=>?@\[\]\\^_`{|}~\t\n\r]', ' ', text)
        words = text.split()

        return words

    def fit_on_texts(self, texts):
        counter = Counter()

        for text in texts:
            counter.update(self._text_to_words(text))

        # Индекс 0 оставлен под пустое значение, индекс 1 — неизвестное слово.
        max_words = self.num_words - 2
        for index, (word, _) in enumerate(counter.most_common(max_words), start=2):
            self.word_index[word] = index

    def texts_to_sequences(self, texts):
        sequences = []

        for text in texts:
            words = self._text_to_words(text)
            sequence = [self.word_index.get(word, 1) for word in words]
            sequences.append(sequence)

        return sequences

# создание токенов
tokenizer = SimpleTokenizer(num_words=vocab_size, lower=True, oov_token='unknown')
tokenizer.fit_on_texts(train_texts)

train_word_indexes = tokenizer.texts_to_sequences(train_texts)
test_word_indexes = tokenizer.texts_to_sequences(test_texts)

print('Размер словаря:', len(tokenizer.word_index))

for i in range(num_classes):
    print(class_names[i])
    print('Обучающая последовательность:', len(train_word_indexes[i]))
    print('Проверочная последовательность:', len(test_word_indexes[i]))
    print()


In [ ]:
def vectorize_sequence(sequence, vocab_size):
    """Преобразование окна текста в вектор bag-of-words."""

    x = np.zeros(vocab_size, dtype=np.float32)

    for word_index in sequence:
        if 0 < word_index < vocab_size:
            x[word_index] = 1.0

    return x

# функция создает обучающую или проверочную выборку
def create_bow_dataset(word_indexes, num_classes, vocab_size, win_size, win_hop):

    # входные данные
    x_data = []

    # правильные ответы: для PyTorch CrossEntropyLoss нужны номера классов, а не one-hot
    y_data = []

    # проходим по каждому классу
    for class_index in range(num_classes):
        sequence = word_indexes[class_index]

        for start in range(0, len(sequence) - win_size, win_hop):
            # берется кусок текста
            window = sequence[start:start + win_size]

            x_data.append(vectorize_sequence(window, vocab_size))
            y_data.append(class_index)

    x_data = np.array(x_data, dtype=np.float32)
    y_data = np.array(y_data, dtype=np.int64)

    return x_data, y_data


In [ ]:
# создание обучающей выборки
x_train, y_train = create_bow_dataset(train_word_indexes, num_classes, vocab_size, win_size, win_hop)

# создание проверочной выборки
x_test, y_test = create_bow_dataset(test_word_indexes, num_classes, vocab_size, win_size, win_hop)

# проверка размеров выборок
print('До балансировки')
print('x_train:', x_train.shape)
print('y_train:', y_train.shape)
print('x_test:', x_test.shape)
print('y_test:', y_test.shape)

# функция для балансировки выборки
def balance_dataset(x_data, y_data):

    # подсчет количества примеров каждого класса
    class_counts = []

    for i in range(num_classes):
        class_counts.append(np.sum(y_data == i))

    # берется минимальное количество примеров среди классов
    min_count = min(class_counts)

    balanced_indexes = []
    random_state = np.random.RandomState(RANDOM_STATE)

    for i in range(num_classes):
        class_indexes = np.where(y_data == i)[0]
        random_state.shuffle(class_indexes)
        class_indexes = class_indexes[:min_count]
        balanced_indexes.extend(class_indexes)

    balanced_indexes = np.array(balanced_indexes)
    random_state.shuffle(balanced_indexes)

    return x_data[balanced_indexes], y_data[balanced_indexes]

# балансировка обучающей выборки
x_train, y_train = balance_dataset(x_train, y_train)

# балансировка проверочной выборки
x_test, y_test = balance_dataset(x_test, y_test)

# проверка размеров выборок
print('\nПосле балансировки')
print('x_train:', x_train.shape)
print('y_train:', y_train.shape)
print('x_test:', x_test.shape)
print('y_test:', y_test.shape)

for i in range(num_classes):
    print(class_names[i])
    print('train:', np.sum(y_train == i))
    print('test:', np.sum(y_test == i))


In [ ]:
# преобразование данных в тензоры PyTorch
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 8

train_loader = DataLoader(
    TensorDataset(x_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(x_test_tensor, y_test_tensor),
    batch_size=batch_size,
    shuffle=False
)

# создание модели
class TeslaReviewClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = TeslaReviewClassifier(vocab_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# функция обучения за одну эпоху
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_batch.size(0)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

# функция проверки
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item() * x_batch.size(0)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

# обучение
epochs = 10
history = {
    'loss': [],
    'accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_accuracy)
    history['val_loss'].append(test_loss)
    history['val_accuracy'].append(test_accuracy)

    print(
        f'Эпоха {epoch + 1}/{epochs} | '
        f'loss: {train_loss:.4f} | accuracy: {train_accuracy:.4f} | '
        f'val_loss: {test_loss:.4f} | val_accuracy: {test_accuracy:.4f}'
    )

# оценка
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)

print('Ошибка на проверочной выборке:', test_loss)
print('Точность на проверочной выборке:', test_accuracy)

# графики обучения и ошибки
plt.figure(figsize=(10, 4))

plt.plot(history['accuracy'], label='Точность на обучающей выборке')
plt.plot(history['val_accuracy'], label='Точность на проверочной выборке')

plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 4))

plt.plot(history['loss'], label='Ошибка на обучающей выборке')
plt.plot(history['val_loss'], label='Ошибка на проверочной выборке')

plt.xlabel('Эпоха')
plt.ylabel('Ошибка')
plt.legend()
plt.grid()
plt.show()
